<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/17-deep-learning-pytorch/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 17: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_count_parameters(f):
    shared = nn.Linear(4, 4)
    frozen = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 2))
    for p in frozen[0].parameters():
        p.requires_grad = False
    mlp = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 10))
    _scenarios("count_parameters", [
        ("the example", lambda: f(nn.Linear(3, 2)), 8, "weights plus biases"),
        ("the lesson's 64-128-128-10 network", lambda: f(mlp), 26122, None),
        ("frozen layers are skipped", lambda: f(frozen), 12, "check p.requires_grad"),
        ("trainable_only=False counts everything", lambda: f(frozen, trainable_only=False), 67, None),
        ("a layer used twice is counted once", lambda: f(nn.Sequential(shared, nn.ReLU(), shared)), 20,
         "model.parameters() already skips duplicates"),
        ("batch norm buffers are not parameters", lambda: f(nn.BatchNorm1d(10)), 20, None),
        ("returns a Python int", lambda: type(f(nn.Linear(3, 2))).__name__, "int", None),
    ])


def check_accuracy_from_logits(f):
    g = torch.Generator().manual_seed(0)
    logits = torch.randn(40, 3, generator=g)
    targets = torch.randint(0, 3, (40,), generator=g)
    expected = float((logits.argmax(dim=1) == targets).double().mean())
    _scenarios("accuracy_from_logits", [
        ("the example", lambda: f(torch.tensor([[2.0, 1.0], [0.0, 3.0], [1.0, 0.5]]), torch.tensor([0, 1, 1])), 2 / 3,
         "the predicted class is the argmax of each row"),
        ("a random batch of 40", lambda: f(logits, targets), expected, None),
        ("negative logits", lambda: f(torch.tensor([[-5.0, -1.0, -3.0]]), torch.tensor([1])), 1.0, None),
        ("returns a Python float", lambda: type(f(logits, targets)).__name__, "float", "use .item()"),
        ("different lengths raise", lambda: f(logits, targets[:10]), Raises(ValueError), None),
    ])


def check_stable_softmax(f):
    g = torch.Generator().manual_seed(1)
    batch = torch.randn(5, 4, generator=g, dtype=torch.float64) * 3
    cube = torch.randn(2, 3, 4, generator=g, dtype=torch.float64)
    mixed = torch.tensor([[0.0, 1.0], [1000.0, 1001.0]], dtype=torch.float64)
    _scenarios("stable_softmax", [
        ("a single vector", lambda: f(torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)),
         torch.softmax(torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64), dim=0), None),
        ("a batch, one softmax per row", lambda: f(batch), torch.softmax(batch, dim=-1), "work along the last dimension"),
        ("large scores don't overflow", lambda: f(torch.tensor([1000.0, 1001.0])),
         torch.softmax(torch.tensor([1000.0, 1001.0]), dim=0), "subtract the largest value first"),
        ("very negative scores", lambda: f(torch.tensor([-1000.0, -1000.0])), torch.tensor([0.5, 0.5]), None),
        ("rows on very different scales", lambda: f(mixed), torch.softmax(mixed, dim=-1),
         "subtract the largest value of each row, not of the whole tensor"),
        ("three dimensions", lambda: f(cube), torch.softmax(cube, dim=-1), None),
    ])


def check_warmup_cosine_lr(f):
    weight = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.SGD([weight], lr=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=0.001)
    torch_rates = []
    for _ in range(20):
        torch_rates.append(optimizer.param_groups[0]["lr"])
        optimizer.step()
        scheduler.step()
    _scenarios("warmup_cosine_lr", [
        ("the first warmup step", lambda: f(0, 100, 0.1, warmup_steps=10), 0.01, "base_lr * (step + 1) / warmup_steps"),
        ("the end of warmup", lambda: f(9, 100, 0.1, warmup_steps=10), 0.1, None),
        ("the start of the decay", lambda: f(10, 100, 0.1, warmup_steps=10), 0.1, None),
        ("halfway through the decay", lambda: f(55, 100, 0.1, warmup_steps=10), 0.05, None),
        ("matches CosineAnnealingLR without warmup", lambda: [f(s, 20, 0.1, min_lr=0.001) for s in range(20)], torch_rates, None),
        ("stays at min_lr after the end", lambda: f(150, 100, 0.1, warmup_steps=10, min_lr=0.001), 0.001,
         "cap progress at 1"),
        ("returns a float", lambda: type(f(3, 20, 1, warmup_steps=5)).__name__, "float", None),
        ("warmup as long as training raises", lambda: f(0, 10, 0.1, warmup_steps=10), Raises(ValueError), None),
    ])


def check_gradient_at(f):
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    original = x.copy()
    A = np.array([[2.0, 1.0], [1.0, 3.0]])

    def unchanged():
        f(lambda t: (t ** 2).sum(), x)
        return x

    _scenarios("gradient_at", [
        ("the example", lambda: f(lambda t: (t ** 2).sum(), [1.0, 2.0, 3.0]), np.array([2.0, 4.0, 6.0]), None),
        ("a product", lambda: f(lambda t: t[0] * t[1] ** 3, [2.0, 1.5]), np.array([3.375, 13.5]), None),
        ("sin and exp", lambda: f(lambda t: torch.sin(t).sum() + torch.exp(t[0]), [0.5, 1.0]),
         np.array([math.cos(0.5) + math.exp(0.5), math.cos(1.0)]), None),
        ("a quadratic form", lambda: f(lambda t: t @ torch.tensor(A) @ t, [1.0, -1.0]), (A + A.T) @ np.array([1.0, -1.0]), None),
        ("keeps the shape of x", lambda: f(lambda t: (t ** 2).sum(), x), 2 * x, None),
        ("float64 precision", lambda: f(lambda t: ((t - 1e8) ** 2).sum(), [1e8 + 1]), np.array([2.0]),
         "in float32, 1e8 + 1 rounds to 1e8"),
        ("returns float64", lambda: str(f(lambda t: t.sum(), [1, 2]).dtype), "float64", None),
        ("x is not changed", unchanged, original, None),
        ("a vector output raises", lambda: f(lambda t: t * 2, [1.0, 2.0]), Raises(ValueError), None),
    ])


def check_make_mlp(f):
    def kinds(model):
        return [type(layer).__name__ for layer in model]

    def shape_out():
        torch.manual_seed(0)
        return tuple(f([64, 128, 128, 10])(torch.randn(5, 64)).shape)

    def sizes_of(model):
        return [(layer.in_features, layer.out_features) for layer in model if isinstance(layer, nn.Linear)]

    _scenarios("make_mlp", [
        ("returns an nn.Sequential", lambda: type(f([4, 8, 3])).__name__, "Sequential", None),
        ("one hidden layer", lambda: kinds(f([4, 8, 3])), ["Linear", "ReLU", "Linear"], "no ReLU after the last Linear"),
        ("layer sizes", lambda: sizes_of(f([64, 128, 128, 10])), [(64, 128), (128, 128), (128, 10)], None),
        ("number of parameters", lambda: sum(p.numel() for p in f([64, 128, 128, 10]).parameters()), 26122, None),
        ("output shape", shape_out, (5, 10), None),
        ("with dropout", lambda: kinds(f([4, 8, 8, 3], dropout=0.5)),
         ["Linear", "ReLU", "Dropout", "Linear", "ReLU", "Dropout", "Linear"], "a Dropout after every ReLU"),
        ("the dropout probability", lambda: [layer.p for layer in f([4, 8, 3], dropout=0.3) if isinstance(layer, nn.Dropout)], [0.3], None),
        ("no hidden layer", lambda: kinds(f([3, 2], dropout=0.5)), ["Linear"], None),
        ("a single size raises", lambda: f([5]), Raises(ValueError), None),
    ])


def check_cross_entropy_from_logits(f):
    g = torch.Generator().manual_seed(2)
    logits = torch.randn(8, 5, generator=g, dtype=torch.float64) * 2
    targets = torch.randint(0, 5, (8,), generator=g)
    big = torch.tensor([[1000.0, 0.0, -1000.0], [0.0, 2000.0, 10.0]], dtype=torch.float64)
    big_targets = torch.tensor([2, 1])

    def grad_of(loss_function):
        x = logits.clone().requires_grad_(True)
        loss_function(x, targets).backward()
        return x.grad

    _scenarios("cross_entropy_from_logits", [
        ("the example", lambda: f(torch.tensor([[2.0, 0.0]]), torch.tensor([0])), torch.tensor(math.log(1 + math.exp(-2.0))),
         "logsumexp(row) - row[true class]; return a tensor"),
        ("a random batch", lambda: f(logits, targets), F.cross_entropy(logits, targets), "the mean over examples"),
        ("returns a single value", lambda: tuple(f(logits, targets).shape), (), None),
        ("large logits stay finite", lambda: f(big, big_targets), F.cross_entropy(big, big_targets),
         "work with logs throughout; don't take the log of a softmax"),
        ("the gradient matches PyTorch's", lambda: grad_of(f), grad_of(F.cross_entropy), None),
    ])


def _toy_data(n=50, seed=3):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 4, generator=g)
    y = (X[:, 0] + X[:, 1] > 0).long() + (X[:, 2] > 1).long()
    return X, y


def _toy_model(dropout=0.0):
    torch.manual_seed(0)
    layers = [nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU()]
    if dropout:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(8, 3))
    return nn.Sequential(*layers)


def _reference_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, count = 0.0, 0
    for X, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
        count += len(y)
    return total / count


def _snapshot(model):
    return [t.detach().clone() for t in list(model.parameters()) + list(model.buffers()) if t.dtype.is_floating_point]


def check_train_one_epoch(f):
    X, y = _toy_data()

    def run(train_function, epochs=1):
        model = _toy_model()
        model.eval()
        loader = DataLoader(TensorDataset(X, y), batch_size=16)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
        losses = [train_function(model, loader, nn.CrossEntropyLoss(), optimizer) for _ in range(epochs)]
        return losses, _snapshot(model)

    expected_losses, expected_state = run(_reference_epoch, epochs=2)
    _scenarios("train_one_epoch", [
        ("the average loss of one epoch", lambda: run(f)[0][0], expected_losses[0],
         "weight each batch's loss by its number of examples"),
        ("returns a Python float", lambda: type(run(f)[0][0]).__name__, "float", "use loss.item()"),
        ("the weights and batch norm statistics after two epochs", lambda: run(f, epochs=2)[1], expected_state,
         "zero the gradients for every batch, and call model.train() first"),
        ("the loss of the second epoch", lambda: run(f, epochs=2)[0][1], expected_losses[1], None),
    ])


def check_evaluate_model(f):
    X, y = _toy_data(n=50, seed=4)
    trained = _toy_model(dropout=0.5)
    optimizer = torch.optim.Adam(trained.parameters(), lr=0.01)
    train_loader = DataLoader(TensorDataset(*_toy_data(n=200, seed=5)), batch_size=32)
    for _ in range(3):
        _reference_epoch(trained, train_loader, nn.CrossEntropyLoss(), optimizer)
    state = {k: v.clone() for k, v in trained.state_dict().items()}

    def fresh():
        trained.load_state_dict(state)
        trained.train()
        return trained

    loader = DataLoader(TensorDataset(X, y), batch_size=16)
    with torch.no_grad():
        model = fresh().eval()
        logits = model(X)
        expected = (F.cross_entropy(logits, y).item(), (logits.argmax(dim=1) == y).double().mean().item())

    def state_after():
        model = fresh()
        f(model, loader, nn.CrossEntropyLoss())
        return _snapshot(model)

    def graph_built():
        seen = []

        def recording_loss(logits, targets):
            seen.append(logits.requires_grad)
            return F.cross_entropy(logits, targets)

        f(fresh(), loader, recording_loss)
        return any(seen)

    def types():
        result = f(fresh(), loader, nn.CrossEntropyLoss())
        return [type(result).__name__] + [type(v).__name__ for v in result]

    _scenarios("evaluate_model", [
        ("loss and accuracy", lambda: f(fresh(), loader, nn.CrossEntropyLoss()), expected,
         "call model.eval(), and weight each batch by its size"),
        ("returns a tuple of two floats", types, ["tuple", "float", "float"], None),
        ("weights and batch norm statistics unchanged", state_after, _snapshot(fresh()), "evaluation mode stops the running statistics from updating"),
        ("no gradients are tracked", graph_built, False, "use torch.no_grad()"),
    ])


def check_sgd_momentum_step(f):
    g = torch.Generator().manual_seed(6)
    shapes = [(3, 2), (4,)]
    start = [torch.randn(s, generator=g, dtype=torch.float64) for s in shapes]
    gradients = [[torch.randn(s, generator=g, dtype=torch.float64) for s in shapes] for _ in range(5)]

    def with_torch(steps, momentum, weight_decay, lr=0.05):
        params = [p.clone().requires_grad_(True) for p in start]
        optimizer = torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
        for grads in gradients[:steps]:
            for p, grad in zip(params, grads):
                p.grad = grad.clone()
            optimizer.step()
        buffers = [optimizer.state[p].get("momentum_buffer") for p in params]
        return [p.detach() for p in params], buffers

    def with_yours(steps, momentum, weight_decay, lr=0.05):
        params = [p.clone() for p in start]
        velocities = [torch.zeros_like(p) for p in start]
        for grads in gradients[:steps]:
            f(params, [grad.clone() for grad in grads], velocities, lr, momentum=momentum, weight_decay=weight_decay)
        return params, velocities

    _scenarios("sgd_momentum_step", [
        ("one step from rest is plain SGD", lambda: with_yours(1, 0.9, 0.0)[0], [p - 0.05 * grad for p, grad in zip(start, gradients[0])],
         "update the tensors in place, for example with p.sub_(...)"),
        ("five steps match torch.optim.SGD", lambda: with_yours(5, 0.9, 0.0)[0], with_torch(5, 0.9, 0.0)[0], None),
        ("the velocities match PyTorch's momentum buffers", lambda: with_yours(5, 0.9, 0.0)[1], with_torch(5, 0.9, 0.0)[1],
         "v = momentum * v + d, with the learning rate applied afterwards"),
        ("with weight decay", lambda: with_yours(5, 0.9, 0.01)[0], with_torch(5, 0.9, 0.01)[0], None),
        ("a different momentum", lambda: with_yours(4, 0.5, 0.0, lr=0.2)[0], with_torch(4, 0.5, 0.0, lr=0.2)[0], None),
    ])


def check_adam_step(f):
    g = torch.Generator().manual_seed(7)
    start = torch.randn(3, 4, generator=g, dtype=torch.float64)
    gradients = [torch.randn(3, 4, generator=g, dtype=torch.float64) for _ in range(10)]
    tiny = [grad * 1e-9 for grad in gradients]

    def with_torch(steps, grads=gradients, lr=0.01, weight_decay=0.0, decoupled=False):
        param = start.clone().requires_grad_(True)
        cls = torch.optim.AdamW if decoupled else torch.optim.Adam
        optimizer = cls([param], lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay)
        for grad in grads[:steps]:
            param.grad = grad.clone()
            optimizer.step()
        state = optimizer.state[param]
        return param.detach(), state["exp_avg"], state["exp_avg_sq"]

    def with_yours(steps, grads=gradients, lr=0.01, weight_decay=0.0, decoupled=False):
        param, m, v = start.clone(), torch.zeros_like(start), torch.zeros_like(start)
        for t, grad in enumerate(grads[:steps], start=1):
            f(param, grad.clone(), m, v, t, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, decoupled=decoupled)
        return param, m, v

    _scenarios("adam_step", [
        ("the first step moves each weight by about lr", lambda: with_yours(1)[0], with_torch(1)[0],
         "bias correction makes the first step lr * g / (|g| + eps)"),
        ("ten steps match torch.optim.Adam", lambda: with_yours(10)[0], with_torch(10)[0], None),
        ("m and v match PyTorch's running averages", lambda: with_yours(10)[1:], with_torch(10)[1:], None),
        ("tiny gradients, where eps matters", lambda: with_yours(5, grads=tiny)[0], with_torch(5, grads=tiny)[0],
         "add eps after the square root"),
        ("Adam's weight decay", lambda: with_yours(10, weight_decay=0.1)[0], with_torch(10, weight_decay=0.1)[0], None),
        ("AdamW's decoupled weight decay", lambda: with_yours(10, weight_decay=0.1, decoupled=True)[0],
         with_torch(10, weight_decay=0.1, decoupled=True)[0], "shrink the weights directly, before the Adam step"),
        ("t = 0 raises", lambda: f(start.clone(), gradients[0], torch.zeros_like(start), torch.zeros_like(start), 0), Raises(ValueError), None),
    ])


def check_batchnorm1d_forward(f):
    g = torch.Generator().manual_seed(8)
    batches = [torch.randn(6, 3, generator=g, dtype=torch.float64) * 2 + 1 for _ in range(3)]
    gamma0 = torch.tensor([1.5, 0.5, -1.0], dtype=torch.float64)
    beta0 = torch.tensor([0.1, -0.2, 0.3], dtype=torch.float64)

    def torch_layer(momentum=0.1):
        layer = nn.BatchNorm1d(3, momentum=momentum).double()
        with torch.no_grad():
            layer.weight.copy_(gamma0)
            layer.bias.copy_(beta0)
        return layer

    def yours_train(momentum=0.1):
        running_mean, running_var = torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64)
        outputs = [f(x, gamma0, beta0, running_mean, running_var, True, momentum=momentum) for x in batches]
        return outputs, running_mean, running_var

    def torch_train(momentum=0.1):
        layer = torch_layer(momentum)
        outputs = [layer(x).detach() for x in batches]
        return outputs, layer.running_mean, layer.running_var

    def yours_eval():
        _, running_mean, running_var = yours_train()
        return f(batches[0][:2] * 3, gamma0, beta0, running_mean, running_var, False)

    def torch_eval():
        layer = torch_layer()
        for x in batches:
            layer(x)
        layer.eval()
        return layer(batches[0][:2] * 3).detach()

    def eval_stats():
        _, running_mean, running_var = yours_train()
        before = [running_mean.clone(), running_var.clone()]
        f(batches[0], gamma0, beta0, running_mean, running_var, False)
        return [running_mean, running_var], before

    def gradients(fn):
        x = batches[0].clone().requires_grad_(True)
        gamma = gamma0.clone().requires_grad_(True)
        beta = beta0.clone().requires_grad_(True)
        weights = torch.arange(18, dtype=torch.float64).reshape(6, 3)
        (fn(x, gamma, beta) * weights).sum().backward()
        return [x.grad, gamma.grad, beta.grad]

    def yours_fn(x, gamma, beta):
        return f(x, gamma, beta, torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64), True)

    def torch_fn(x, gamma, beta):
        return F.batch_norm(x, torch.zeros(3, dtype=torch.float64), torch.ones(3, dtype=torch.float64), gamma, beta, training=True)

    _scenarios("batchnorm1d_forward", [
        ("training-mode output", lambda: yours_train()[0][0], torch_train()[0][0], "normalise with the biased variance of the batch"),
        ("running statistics after three batches", lambda: list(yours_train()[1:]), list(torch_train()[1:]),
         "the running variance uses the unbiased variance, var * n / (n - 1)"),
        ("a different momentum", lambda: list(yours_train(0.3)[1:]), list(torch_train(0.3)[1:]), None),
        ("evaluation mode uses the running statistics", yours_eval, torch_eval(), None),
        ("evaluation mode leaves the statistics alone", lambda: eval_stats()[0], eval_stats()[1], None),
        ("gradients flow through the batch statistics", lambda: gradients(yours_fn), gradients(torch_fn),
         "don't detach the batch mean or variance"),
        ("a batch of one raises", lambda: f(batches[0][:1], gamma0, beta0, torch.zeros(3, dtype=torch.float64),
                                            torch.ones(3, dtype=torch.float64), True), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `count_parameters`

In [2]:
def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

In [3]:
check_count_parameters(count_parameters)

PASS  all 7 tests passed for count_parameters


`model.parameters()` walks through every layer inside the model, and `numel()` gives the number of elements in each tensor. Summing them gives the total.

Three details the checks look at:

- **Shared layers are counted once.** If the same `nn.Linear` object appears twice in a model, its weights exist once in memory and are updated once. `model.parameters()` already skips duplicates. Code that loops over the layers of an `nn.Sequential` (`for layer in model`) and adds up each layer's weights counts a shared layer twice.
- **Buffers aren't parameters.** Batch normalisation keeps a running mean and variance, but those are *buffers*: saved with the model, never trained. `nn.BatchNorm1d(10)` has 20 parameters (a scale and a shift per feature).
- **Frozen parameters.** Setting `requires_grad = False` is how you freeze part of a network, which you'll do for transfer learning in Module 18. Printing both counts is a quick check that freezing worked.

### Exercise 2: `accuracy_from_logits`

In [4]:
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    if len(logits) != len(targets):
        raise ValueError("logits and targets must have the same length")
    predictions = logits.argmax(dim=1)
    return (predictions == targets).float().mean().item()

In [5]:
check_accuracy_from_logits(accuracy_from_logits)

PASS  all 5 tests passed for accuracy_from_logits


`argmax(dim=1)` finds the largest score **within each row**, which is one prediction per example. With `dim=0` you'd get the example with the largest score for each class, which has a different shape and means nothing useful; the check uses more examples than classes, so the shapes don't line up by accident.

There's no need for softmax first. Softmax never changes which score is largest, so the argmax of the logits and the argmax of the probabilities are the same.

Comparing gives a tensor of `True`/`False`; `.float().mean()` turns it into a fraction, and `.item()` turns a one-element tensor into a plain Python number. Returning a tensor instead of a float is a common source of slow training loops: adding tensors to a running total keeps them alive, and on a GPU each one is a small transfer.

### Exercise 3: `stable_softmax`

In [6]:
def stable_softmax(logits: torch.Tensor) -> torch.Tensor:
    shifted = logits - logits.max(dim=-1, keepdim=True).values
    exp = shifted.exp()
    return exp / exp.sum(dim=-1, keepdim=True)

In [7]:
check_stable_softmax(stable_softmax)

PASS  all 6 tests passed for stable_softmax


`max(dim=-1, keepdim=True)` gives the largest value of each row, with the reduced dimension kept as size 1 so that the subtraction broadcasts row by row. Without `keepdim=True`, a `(4, 3)` tensor minus a `(4,)` tensor either fails or, for a square input, silently subtracts the wrong numbers.

The shift has to be **per row**. Subtracting the single largest value in the whole tensor also avoids overflow, but a row whose scores are all far below that value underflows instead: every `exp` becomes 0 and the row turns into `0 / 0 = nan`. One of the checks has two rows on very different scales to catch this.

After the shift, the largest value in each row is 0, so its `exp` is exactly 1 and the sum is at least 1. That's why the division is always safe. PyTorch's `softmax`, `log_softmax` and `cross_entropy` all use this trick internally.

### Exercise 4: `warmup_cosine_lr`

In [8]:
def warmup_cosine_lr(step: int, total_steps: int, base_lr: float, warmup_steps: int = 0, min_lr: float = 0.0) -> float:
    if warmup_steps >= total_steps:
        raise ValueError("warmup_steps must be smaller than total_steps")
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = min((step - warmup_steps) / (total_steps - warmup_steps), 1.0)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [9]:
check_warmup_cosine_lr(warmup_cosine_lr)

PASS  all 8 tests passed for warmup_cosine_lr


At the start of the decay, `progress` is 0, `cos(0) = 1`, and the rate is `base_lr`. At the end, `progress` is 1, `cos(pi) = -1`, and the rate is `min_lr`. In between, the cosine gives a curve that is flat at both ends: it stays near the full rate for a while, falls fastest in the middle, and spends a long time at small rates at the end, which is where the fine adjustments happen.

Warmup uses `step + 1` so that the very first step already has a small positive rate (a rate of exactly 0 would waste a step), and the last warmup step reaches `base_lr`. Warmup matters most with Adam and large models: early on, Adam's estimates of the gradient size come from very few steps, and big early updates can push the network into a bad region it never recovers from.

The cap at 1 keeps the rate at `min_lr` if training runs past `total_steps`. Without it, the cosine would start rising again.

To use a function like this in PyTorch, wrap it in `torch.optim.lr_scheduler.LambdaLR`, which multiplies the optimiser's rate by whatever your function returns for each step (so you'd divide by `base_lr`).

## Medium

### Exercise 5: `gradient_at`

In [10]:
def gradient_at(f, x) -> np.ndarray:
    point = torch.tensor(np.asarray(x, dtype=np.float64), requires_grad=True)
    value = f(point)
    if value.numel() != 1:
        raise ValueError("f must return a single number")
    value.backward()
    return point.grad.numpy()

In [11]:
check_gradient_at(gradient_at)

PASS  all 9 tests passed for gradient_at


`requires_grad=True` tells PyTorch to record every operation done with `point`. Calling `backward()` on the result walks that record backwards, applying the chain rule, and stores the gradient in `point.grad`.

`torch.tensor(...)` **copies** the data, so the caller's array is never touched. `torch.from_numpy` would share memory with it (Section 2 of the lesson), which is fine for reading but a trap if anything writes to the tensor.

Precision is why the exercise insists on `float64`. PyTorch's default is `float32`, which keeps about 7 significant digits. One check differentiates `(t - 1e8) ** 2` at `t = 1e8 + 1`: in `float32`, `1e8 + 1` rounds to `1e8` and the gradient comes out as 0 instead of 2. Neural networks train fine in `float32` (and even 16-bit formats), because their numbers are small and their gradients are noisy anyway; numerical checks like this one need the extra digits.

`backward()` only works on a single number, because a gradient is the derivative of **one** output. For a vector output you'd need a Jacobian (`torch.autograd.functional.jacobian`), or to reduce it to one number first, which is exactly what a loss function does.

### Exercise 6: `make_mlp`

In [12]:
def make_mlp(sizes, dropout: float = 0.0) -> nn.Sequential:
    if len(sizes) < 2:
        raise ValueError("sizes needs at least an input size and an output size")
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)

In [13]:
check_make_mlp(make_mlp)

PASS  all 9 tests passed for make_mlp


Building a list of layers and unpacking it into `nn.Sequential(*layers)` is the usual way to make a network whose depth is a setting. Note that a plain Python list of layers is **not** enough on its own: PyTorch only finds the parameters of layers stored on a module (here, inside the `Sequential`). For a list stored as an attribute of your own `nn.Module`, use `nn.ModuleList`.

The last layer has no ReLU because the output feeds into `nn.CrossEntropyLoss`, which expects raw logits that can be negative. A ReLU there would clip every negative logit to zero, and the network could never say "definitely not this class" with more than a zero.

Dropout goes after the activation, so it randomly zeroes hidden units during training. There's no dropout after the output layer: zeroing a logit would simply corrupt the prediction.

### Exercise 7: `cross_entropy_from_logits`

In [14]:
def cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    log_probs = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    return -log_probs[torch.arange(len(targets)), targets].mean()

In [15]:
check_cross_entropy_from_logits(cross_entropy_from_logits)

PASS  all 5 tests passed for cross_entropy_from_logits


`logits - logsumexp(logits)` is the log of the softmax, computed without ever forming the probabilities. Then `log_probs[torch.arange(n), targets]` picks one value per row: row 0 at column `targets[0]`, row 1 at column `targets[1]`, and so on. `log_probs.gather(1, targets.unsqueeze(1))` does the same thing.

The tempting version, `-torch.log(torch.softmax(logits, dim=1))`, has two problems. Without the max trick, the softmax overflows. With it, a confident wrong answer can give a probability that rounds to exactly 0, and `log(0)` is minus infinity. Working with logs from the start avoids both, which is why PyTorch's loss takes logits rather than probabilities.

The gradient of this loss with respect to the logits is `(softmax(logits) - one_hot(targets)) / n`: predicted probability minus truth, the same simple form as in logistic regression (Module 12). The checks compare your gradient with PyTorch's, which also catches using the sum instead of the mean.

### Exercise 8: `train_one_epoch`

In [16]:
def train_one_epoch(model: nn.Module, loader, loss_fn, optimizer) -> float:
    model.train()
    total, count = 0.0, 0
    for X, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
        count += len(y)
    return total / count

In [17]:
check_train_one_epoch(train_one_epoch)

PASS  all 4 tests passed for train_one_epoch


These five lines (zero, forward, loss, backward, step) are the heart of every PyTorch training script, from this small network to the largest language models.

What each check catches:

- **Forgetting `zero_grad()`.** PyTorch adds new gradients to the old ones (Section 3 of the lesson), so without it each step uses the sum of all gradients so far. Training still runs, and the loss may even fall for a while, which is what makes this bug hard to spot. The weights after one epoch don't match the reference.
- **Forgetting `model.train()`.** The check hands you a model that was left in evaluation mode. Its batch normalisation layer then uses the stored running statistics instead of the batch statistics and never updates them.
- **Averaging the batch losses equally.** With 50 examples in batches of 16, the last batch has only 2 examples. A plain mean of the four batch losses gives that batch the same weight as the others.
- **`total += loss`** instead of `loss.item()`. This keeps every batch's computation graph alive in memory, and returns a tensor instead of a float.

The returned loss is an average over an epoch in which the weights kept changing, so it's a little pessimistic compared to the loss of the final weights. That's normal, and it's why the training curve in a plot often sits slightly above where you'd expect.

### Exercise 9: `evaluate_model`

In [18]:
@torch.no_grad()
def evaluate_model(model: nn.Module, loader, loss_fn):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for X, y in loader:
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        count += len(y)
    return total_loss / count, correct / count

In [19]:
check_evaluate_model(evaluate_model)

PASS  all 4 tests passed for evaluate_model


`model.eval()` and `torch.no_grad()` do different jobs, and you need both:

- `model.eval()` changes how some **layers** behave. Dropout stops dropping, and batch normalisation switches from batch statistics to the running averages it collected during training. Without it, the same model gives different answers each time you evaluate it, and batch norm's running statistics drift towards the validation data.
- `torch.no_grad()` turns off the recording of operations for autograd. The answers are the same, but it saves memory and time, which matters a lot for large models or large validation sets.

One check passes in a loss function that records whether the logits it receives require gradients; inside `no_grad` they don't.

Correct predictions are counted and divided by the total at the end, rather than averaging per-batch accuracies, for the same reason as the loss: the last batch may be smaller. Remember to call `model.train()` before training again; the `train_one_epoch` from the previous exercise does that for you.

## Hard

### Exercise 10: `sgd_momentum_step`

In [20]:
def sgd_momentum_step(params, grads, velocities, lr: float, momentum: float = 0.9, weight_decay: float = 0.0) -> None:
    with torch.no_grad():
        for p, g, v in zip(params, grads, velocities):
            d = g + weight_decay * p
            v.mul_(momentum).add_(d)
            p.sub_(lr * v)

In [21]:
check_sgd_momentum_step(sgd_momentum_step)

PASS  all 5 tests passed for sgd_momentum_step


The velocity is a running sum of past gradients, each one shrunk by `momentum` per step. With `momentum = 0.9`, a gradient keeps influencing the updates for roughly 10 steps. Directions where the gradient keeps the same sign build up speed, and directions where it flips back and forth cancel out, which is why momentum moves faster along long, narrow valleys in the loss.

**In place matters.** `p = p - lr * v` creates a new tensor and points the local name `p` at it; the tensor in the caller's list never changes. `p.sub_(lr * v)` modifies the tensor itself. Real optimisers work this way so the model's parameters stay the same objects. `torch.no_grad()` is there because changing a parameter that requires gradients in place is otherwise an error.

**Two textbook versions.** Many books write `v = momentum * v + lr * d` and then `p = p - v`, putting the learning rate inside the velocity. With a fixed learning rate the weights come out identical, but the velocity is `lr` times bigger, and the two versions behave differently as soon as the learning rate changes during training. PyTorch uses the version in this exercise, and the check compares your velocities with PyTorch's own momentum buffers to make sure you do too.

Weight decay here is added to the gradient, which is the same as adding `weight_decay / 2 * ||w||^2` to the loss: L2 regularisation, as in ridge regression (Module 12).

### Exercise 11: `adam_step`

In [22]:
def adam_step(param, grad, m, v, t: int, lr: float = 1e-3, betas=(0.9, 0.999), eps: float = 1e-8,
              weight_decay: float = 0.0, decoupled: bool = False) -> None:
    if t < 1:
        raise ValueError("t counts steps from 1")
    beta1, beta2 = betas
    with torch.no_grad():
        if decoupled:
            param.mul_(1 - lr * weight_decay)
            g = grad
        else:
            g = grad + weight_decay * param
        m.mul_(beta1).add_((1 - beta1) * g)
        v.mul_(beta2).add_((1 - beta2) * g * g)
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        param.sub_(lr * m_hat / (v_hat.sqrt() + eps))

In [23]:
check_adam_step(adam_step)

PASS  all 7 tests passed for adam_step


Adam keeps two running averages per weight: `m`, the average gradient (momentum), and `v`, the average squared gradient (how large the gradients for this weight usually are). The update divides one by the square root of the other, so every weight moves by roughly `lr` per step whatever the scale of its gradients. That's why Adam needs much less tuning than SGD, and why `lr=1e-3` works for so many problems.

**Bias correction.** Because `m` and `v` start at zero, their early values are far too small: after one step, `m` is only `0.1 * g`. Dividing by `1 - beta ** t` fixes this exactly. On the first step, `m_hat = g` and `v_hat = g * g`, so the update is `lr * g / (|g| + eps)`, about `lr` in the direction of `-g`. Without bias correction the first steps would be the wrong size, which one of the checks tests.

**Where eps goes.** `eps` is added **after** the square root. Putting it inside (`sqrt(v_hat + eps)`) only matters when gradients are tiny, and one check uses gradients of about `1e-9` to make the difference visible.

**Adam vs AdamW.** In Adam, weight decay is added to the gradient and then divided by `sqrt(v_hat)` like everything else, so weights with large gradients are barely regularised. AdamW (Loshchilov and Hutter, 2019) applies the decay directly to the weights, separately from the adaptive step. That is the version used to train most modern networks, and it's why you should normally reach for `torch.optim.AdamW` when you want weight decay.

### Exercise 12: `batchnorm1d_forward`

In [24]:
def batchnorm1d_forward(x, gamma, beta, running_mean, running_var, training: bool,
                        momentum: float = 0.1, eps: float = 1e-5) -> torch.Tensor:
    if training:
        n = x.shape[0]
        if n < 2:
            raise ValueError("training mode needs more than one example per batch")
        mean = x.mean(dim=0)
        var = x.var(dim=0, correction=0)
        with torch.no_grad():
            running_mean.mul_(1 - momentum).add_(momentum * mean)
            running_var.mul_(1 - momentum).add_(momentum * var * n / (n - 1))
    else:
        mean, var = running_mean, running_var
    return gamma * (x - mean) / torch.sqrt(var + eps) + beta

In [25]:
check_batchnorm1d_forward(batchnorm1d_forward)

PASS  all 7 tests passed for batchnorm1d_forward


Batch normalisation rescales each feature to mean 0 and variance 1 **within the batch**, then lets the network choose a new scale and shift through `gamma` and `beta`. Keeping the inputs of every layer on a steady scale is what let the 10-layer network in Section 10 of the lesson train at all.

**Two kinds of variance.** The forward pass normalises with the biased variance (divide by `n`), which describes this batch exactly. The running estimate, meant to describe the whole data set, uses the unbiased version (divide by `n - 1`). `var * n / (n - 1)` converts one into the other. Mixing them up changes the results only slightly for large batches, which is why the check uses a batch of 6.

**Gradients through the statistics.** The mean and variance depend on every example in the batch, so the gradient for one example depends on the others. Detaching `mean` or `var` gives a simpler but wrong gradient, and the check compares your gradient with PyTorch's. The running statistics, on the other hand, are bookkeeping, so they're updated inside `torch.no_grad()`.

**Why a batch of 1 fails.** The variance of a single value is 0 (and the unbiased one is undefined), so every output would be just `beta`. PyTorch raises an error for the same reason, and that's why a `DataLoader` for batch norm models sometimes uses `drop_last=True`.

In evaluation mode the network uses fixed statistics, so each example's output no longer depends on the rest of the batch. This is why forgetting `model.eval()` makes predictions change with the batch they happen to be in.

## What's next

The mini project in `04_mini_project.ipynb` uses these pieces on real images: Fashion-MNIST, 70,000 small photos of clothing. You'll build a baseline, train an MLP, diagnose overfitting, add dropout, weight decay and a learning-rate schedule, and look closely at the mistakes the final model makes.